### Generates scatter plots to visualize solver performance.

This script queries a benchmark database for completed runs and generates two scatter plots on a log-log scale:
1. Solver time vs. the number of edges in the problem.
2. Number of IPM iterations vs. the number of edges in the problem.

Each solver is represented by a different color in the plots.

In [1]:
import sqlite3
import polars as pl
import matplotlib.pyplot as plt
import argparse

In [2]:
def plot_solver_performance(db_path):
    conn = sqlite3.connect(db_path)
    query = """
    SELECT
        p.num_edges,
        r.time_s,
        r.iters,
        r.solver_name
    FROM
        runs r
    JOIN
        problems p ON r.problem_id = p.id
    JOIN
        configs c ON r.config_id = c.id
    WHERE
        r.status = 'Trm_Optimal'
    ORDER BY
        p.name, r.solver_name, sh.ipm_iter
    """
    df = pl.read_database(query, conn)
    conn.close()

    print("DataFrame head:\n", df.head())

    fig, axes = plt.subplots(2, 1, figsize=(10, 12))

    for solver_key, group in df.group_by("solver_name"):
        solver_name = solver_key[0]
        axes[0].scatter(
            group["num_edges"]
            , group["time_s"]
            , label=solver_name
            , alpha=0.7
        )

    axes[0].set_xscale("log")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Number of Edges (log scale)")
    axes[0].set_ylabel("Time (s) (log scale)")
    axes[0].set_title("Solver Performance: Time vs. Number of Edges")
    axes[0].legend()
    axes[0].grid(True, which="both", ls="--")

    for solver_key, group in df.group_by("solver_name"):
        solver_name = solver_key[0]
        axes[1].scatter(
            group["num_edges"]
            , group["iters"]
            , label=solver_name
            , alpha=0.7
        )

    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Number of Edges (log scale)")
    axes[1].set_ylabel("Iterations (log scale)")
    axes[1].set_title("Solver Performance: Iterations vs. Number of Edges")
    axes[1].legend()
    axes[1].grid(True, which="both", ls="--")

    plt.tight_layout()
    plt.show()

plot_solver_performance("lemon_gridwide.db") # Assuming a default db file for demonstration


OperationalError: no such table: runs